In [ ]:
# Importo las librerias necesarias para descompresión de archivos para
# manipulación de tablas para expresiones regulares y para lectura y
# parseo de archivos XML:
import zipfile
import os
import pandas as pd
import os
import xml.etree.ElementTree as ET
import re

# Una vez cargado el file en sample_data trabajando en google colab con click
# derecho me dice la ruta o path y le indico de donde lo tiene que tomar y ext
zip_path = "/content/2023-xml.zip"
extract_path = "/content/xml_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP descomprimido", extract_path)


# Creo un diccionario vacío donde se almacenará cada contrato o licitación
registros = []

# Creo una función para limpiar texto donde cada registro vacío será un N/A
# limpio comas, puntos y comas, comillas y puntos finales además de convertir
# todo a mayúsculas.

def limpiar_texto(texto):
    if not texto:
        return "N/A"
    texto = texto.replace(",", "").replace(";", "")
    texto = re.sub(r'["\.]+$', '', texto.strip())
    return texto.upper()

# Esta función que le continua es para sacar la fecha del contrato,
# y estandarizarla a formato YYYY-MM-DD.
def extraer_fecha(nodo):
    if nodo is None or nodo.text is None:
        return None
    return nodo.text[:10]

# Si bien la carpeta tiene que existir porque arriba le indique donde debe
# descomprimirla y es la misma ruta, evito errores de descompresión
ruta_xml = "/content/xml_data"

if not os.path.exists(ruta_xml):
    print(f"WARNING: La carpeta no existe.")
    # Aquí procedo a recorrer todos los xml que haya
else:
    for raiz, _, archivos in os.walk(ruta_xml):
        for nombre_archivo in archivos:
            if nombre_archivo.endswith('.xml'):
              # Auí los leo
                try:
                    tree = ET.parse(os.path.join(raiz, nombre_archivo))
                    root = tree.getroot()

                    # Campos principales que me interesan los pongo como variables
                    # y con {*} igonoro namespaces.
                    titulo = root.find('.//{*}ProcurementProject/{*}Name')
                    importe = root.find('.//{*}BudgetAmount/{*}TaxExclusiveAmount')
                    empresa = root.find('.//{*}WinningParty//{*}PartyName/{*}Name')
                    nif = root.find('.//{*}WinningParty//{*}PartyIdentification/{*}ID')

                    # Con respecto a las fechas vi 2 en el XML una de publicación y se ve que por ley
                    # debe haber una de actualización, prefiero sacar las 2 para poder darle más sentido
                    # a la correlación con la bolsa.
                    fecha_issue = root.find('.//{*}IssueDate')
                    fecha_updated = root.find('.//{*}updated')

                    fecha_adjudicacion = extraer_fecha(fecha_issue)
                    fecha_publicacion = extraer_fecha(fecha_updated)

                    if titulo is not None and importe is not None:
                      # Voy llenando el diccionario con los campos correspondientes
                        registros.append({
                            'Titulo': limpiar_texto(titulo.text),
                            'Importe': float(importe.text),
                            'Empresa': limpiar_texto(empresa.text) if empresa is not None else "NO ADJUDICADO",
                            'NIF': nif.text.strip() if nif is not None else "N/A",
                            'Fecha_Adjudicacion': fecha_adjudicacion,
                            'Fecha_Publicacion': fecha_publicacion
                        })
# Manejo las excepciones para que si un XML falla continue con el siguiente y no se paralice
                except Exception as e:
                    continue

# Convierto el diccionario en una tabla de tipo excel
df = pd.DataFrame(registros)
# Descartando lso vacíos
if df.empty:
    print("No se encontraron registros válidos para procesar")
else:
    # Filtro solo  los contratos adjudicados con diferente !=
    df = df[df['Empresa'] != "NO ADJUDICADO"]

    # Elimino las filas sin fecha
    df = df[df['Fecha_Publicacion'].notna()]

    # Guardo CSV optimizado para Excel en el que pongo de separador punto y coma
    # y los decimales con coma, porque ese ha sido el estándar seguido en otras asigntauras
    df.to_csv(
        'Licitaciones_Andalucia_limpio.csv',
        index=False,
        sep=';',
        decimal=',',
        encoding='utf-8-sig'
    )

    print("Archivo generado con éxito")
    # Veo el total de registros que saco para ver si esta dentro de los requistos mostraos en el enunciado
    print(f"Total registros: {len(df)}")

ZIP descomprimido en: /content/xml_data
¡Archivo generado con éxito!
Total registros: 4631
